In [ ]:
from langchain_community.document_loaders import PyMuPDFLoader, PyPDFLoader
from pathlib import Path

In [ ]:
def process_all_pdf(path_directory):
    all_docs= []
    path_dir= Path(path_directory)
    pdf_files= list(path_dir.glob("**/*.pdf"))

    print(f"Founded {len(pdf_files)} PDF Files")

    for pdf_file in pdf_files:

        print(f"In processing: {pdf_file.name}")

        try:
            loader= PyMuPDFLoader(str(pdf_file))
            documents= loader.load()

            for doc in documents:
                doc.metadata["Source_file"]= pdf_file.name
                doc.metadata["File_type"]="pdf"

            all_docs.extend(documents)

            print(f"Loaded: {len(documents)} pages")

        except Exception as e:
            print(f"Error founded: {e}")
    
    print(f"Total Documents loaded: {len(all_docs)}")

    return all_docs

In [ ]:
pdfs= process_all_pdf(r"C:\Users\Fatema Kanchwala\OneDrive\Attachments\RAG\data")
pdfs

Founded 4 PDF Files
In processing: DS(U1).pdf
Loaded: 41 pages
In processing: DS(U2).pdf
Loaded: 51 pages
In processing: DS(U3).pdf
Loaded: 36 pages
In processing: DS(U4).pdf
Loaded: 37 pages
Total Documents loaded: 165


[Document(metadata={'producer': 'www.ilovepdf.com', 'creator': 'Microsoft® Word 2010', 'creationdate': '2018-07-18T05:28:28+05:30', 'source': 'C:\\Users\\Fatema Kanchwala\\OneDrive\\Attachments\\RAG\\data\\DS(U1).pdf', 'file_path': 'C:\\Users\\Fatema Kanchwala\\OneDrive\\Attachments\\RAG\\data\\DS(U1).pdf', 'total_pages': 41, 'format': 'PDF 1.5', 'title': '', 'author': 'CSELAB', 'subject': '', 'keywords': '', 'moddate': '2019-09-10T13:53:49+00:00', 'trapped': '', 'modDate': 'D:20190910135349Z', 'creationDate': "D:20180718052828+05'30'", 'page': 0, 'Source_file': 'DS(U1).pdf', 'File_type': 'pdf'}, page_content='UNIT I \n \nIntroduction to Algorithm  –  Programming principles  –  Creating programs-  Analyzing  \nprograms.   \nArrays:  One  dimensional  array,  multidimensional  array.  Pointers  -Searching:  Linear  search,  \nBinary  Search.   \nSorting  techniques:  Internal  sorting  -Insertion  Sort,  Selection  Sort,  Shell  Sort,  Bubble  Sort,  \nQuick  Sort,  Merge  Sort  and Rad

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

def split_documents(documents, chunk_size=2000, chunk_overlap=300):
    text_split= RecursiveCharacterTextSplitter(chunk_size= chunk_size, chunk_overlap= chunk_overlap, length_function= len, separators= ["\n\n", "\n", " ", ""])

    split_docs= text_split.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")

    if split_docs:
        print(f"Example Chunk: ")
        print(f"Content: {split_docs[0].page_content[:300]}")
        print(f"Metadata: {split_docs[0].metadata}")

    return split_docs

In [ ]:
chunks= split_documents(pdfs)
chunks

Split 165 documents into 182 chunks
Example Chunk: 
Content: UNIT I 
 
Introduction to Algorithm  –  Programming principles  –  Creating programs-  Analyzing  
programs.   
Arrays:  One  dimensional  array,  multidimensional  array.  Pointers  -Searching:  Linear  search,  
Binary  Search.   
Sorting  techniques:  Internal  sorting  -Insertion  Sort,  Selecti
Metadata: {'producer': 'www.ilovepdf.com', 'creator': 'Microsoft® Word 2010', 'creationdate': '2018-07-18T05:28:28+05:30', 'source': 'C:\\Users\\Fatema Kanchwala\\OneDrive\\Attachments\\RAG\\data\\DS(U1).pdf', 'file_path': 'C:\\Users\\Fatema Kanchwala\\OneDrive\\Attachments\\RAG\\data\\DS(U1).pdf', 'total_pages': 41, 'format': 'PDF 1.5', 'title': '', 'author': 'CSELAB', 'subject': '', 'keywords': '', 'moddate': '2019-09-10T13:53:49+00:00', 'trapped': '', 'modDate': 'D:20190910135349Z', 'creationDate': "D:20180718052828+05'30'", 'page': 0, 'Source_file': 'DS(U1).pdf', 'File_type': 'pdf'}


[Document(metadata={'producer': 'www.ilovepdf.com', 'creator': 'Microsoft® Word 2010', 'creationdate': '2018-07-18T05:28:28+05:30', 'source': 'C:\\Users\\Fatema Kanchwala\\OneDrive\\Attachments\\RAG\\data\\DS(U1).pdf', 'file_path': 'C:\\Users\\Fatema Kanchwala\\OneDrive\\Attachments\\RAG\\data\\DS(U1).pdf', 'total_pages': 41, 'format': 'PDF 1.5', 'title': '', 'author': 'CSELAB', 'subject': '', 'keywords': '', 'moddate': '2019-09-10T13:53:49+00:00', 'trapped': '', 'modDate': 'D:20190910135349Z', 'creationDate': "D:20180718052828+05'30'", 'page': 0, 'Source_file': 'DS(U1).pdf', 'File_type': 'pdf'}, page_content='UNIT I \n \nIntroduction to Algorithm  –  Programming principles  –  Creating programs-  Analyzing  \nprograms.   \nArrays:  One  dimensional  array,  multidimensional  array.  Pointers  -Searching:  Linear  search,  \nBinary  Search.   \nSorting  techniques:  Internal  sorting  -Insertion  Sort,  Selection  Sort,  Shell  Sort,  Bubble  Sort,  \nQuick  Sort,  Merge  Sort  and Rad

In [ ]:
len(chunks)

182

In [ ]:
import numpy as np
from sentence_transformers import SentenceTransformer
from typing import Dict, List, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
class EmbeddingManager:
    def __init__(self, model_name: str= "all-MiniLM-L6-v2"):
        self.model_name= model_name
        self.model= None
        self._load_model()

    def _load_model(self):
        try:
            print(f"Loading Model {self.model_name}")
            self.model= SentenceTransformer(self.model_name)
            print(f"Loaded successfully! Dimension: {self.model.get_embedding_dimension()}")
        except Exception as e:
            print(f"Error Found: {e}")
            raise

    def generate_embeddings(self, text: List[str])-> np.ndarray:
        if not self.model:
            raise ValueError("Model not loaded")
        print(f"Generating Embeddings of {len(text)} texts")

        embeddings= self.model.encode(text)
        print(f"Embedding Done! Shape: {embeddings.shape}")

        return embeddings

In [ ]:
embedding_manager= EmbeddingManager()
embedding_manager

Loading Model all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1584.99it/s]


Loaded successfully! Dimension: 384


In [ ]:
import os 
from chromadb.config import Settings
import chromadb
import uuid

In [ ]:
class VectorStore:

    def __init__(self, collection_name: str= "pdf_documents", persist_directory: str= "../.data"):

        self.collection_name= collection_name
        self.persist_directory= persist_directory
        self.client= None
        self.collection= None
        self._initialize_store()

    def _initialize_store(self):

        try:
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client= chromadb.PersistentClient(path= self.persist_directory)

            self.collection= self.client.get_or_create_collection(
                name= self.collection_name,
                metadata={"Description": "PDF Document embeddings for RAG"}
            )

        except Exception as e:
            print("Error found!")
            raise

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):

        if len(documents)!= len(documents):
            raise ValueError("Match the length of documents with the embeddings")

        ids=[]
        metadatas=[]
        document_texts=[]
        embeddings_array=[]

        for i, (doc,embedding) in enumerate(zip(documents,embeddings)):
            id= f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(id)

            metadata= dict(doc.metadata)
            metadata['doc_index']=i
            metadata['content_length']= len(doc.page_content)
            metadatas.append(metadata)

            document_texts.append(doc.page_content)

            embeddings_array.append(embedding.tolist())

        try:
            self.collection.add(
                ids= ids,
                metadatas= metadatas,
                documents= document_texts,
                embeddings= embeddings_array
            )

        except Exception as e:
            print("Error founf in adding documents!")
            raise

In [ ]:
vector_store= VectorStore()
vector_store

In [ ]:
chunks

[Document(metadata={'producer': 'www.ilovepdf.com', 'creator': 'Microsoft® Word 2010', 'creationdate': '2018-07-18T05:28:28+05:30', 'source': 'C:\\Users\\Fatema Kanchwala\\OneDrive\\Attachments\\RAG\\data\\DS(U1).pdf', 'file_path': 'C:\\Users\\Fatema Kanchwala\\OneDrive\\Attachments\\RAG\\data\\DS(U1).pdf', 'total_pages': 41, 'format': 'PDF 1.5', 'title': '', 'author': 'CSELAB', 'subject': '', 'keywords': '', 'moddate': '2019-09-10T13:53:49+00:00', 'trapped': '', 'modDate': 'D:20190910135349Z', 'creationDate': "D:20180718052828+05'30'", 'page': 0, 'Source_file': 'DS(U1).pdf', 'File_type': 'pdf'}, page_content='UNIT I \n \nIntroduction to Algorithm  –  Programming principles  –  Creating programs-  Analyzing  \nprograms.   \nArrays:  One  dimensional  array,  multidimensional  array.  Pointers  -Searching:  Linear  search,  \nBinary  Search.   \nSorting  techniques:  Internal  sorting  -Insertion  Sort,  Selection  Sort,  Shell  Sort,  Bubble  Sort,  \nQuick  Sort,  Merge  Sort  and Rad

In [ ]:
texts= [doc.page_content for doc in chunks]
texts

['UNIT I \n \nIntroduction to Algorithm  –  Programming principles  –  Creating programs-  Analyzing  \nprograms.   \nArrays:  One  dimensional  array,  multidimensional  array.  Pointers  -Searching:  Linear  search,  \nBinary  Search.   \nSorting  techniques:  Internal  sorting  -Insertion  Sort,  Selection  Sort,  Shell  Sort,  Bubble  Sort,  \nQuick  Sort,  Merge  Sort  and Radix Sort.  \n \n2 MARKS \n1. \nWhat is data structure and algorithm? (April 2011, Nov 2012, April 2013) \n  \n Data Structure is a particular way of storing and organizing data in a computer. so that it can be \nused efficiently. \n  \n It is the conceptual and concrete ways to organize data for efficient storage and efficient manipulation. \n  \nA data structure is a way of organizing data that considers not only the items stored, but also their \nrelationship to each other. Advance knowledge about the relationship between data items allows designing \nof efficient algorithms for the manipulation of data. \n 

In [ ]:
embeddings= embedding_manager.generate_embeddings(texts)
embeddings

Generating Embeddings of 182 texts
Embedding Done! Shape: (182, 384)


array([[ 0.00118235,  0.02817124, -0.00295807, ...,  0.00486339,
        -0.02850987, -0.04571202],
       [ 0.03813074,  0.01018343, -0.00084049, ...,  0.00652404,
         0.01926095, -0.03346094],
       [-0.04183737,  0.03700625, -0.01780623, ...,  0.05881381,
         0.0884438 , -0.04372542],
       ...,
       [-0.03337651,  0.03627374,  0.06589917, ...,  0.03007171,
        -0.01004769, -0.01102582],
       [ 0.01287587,  0.04522741,  0.04215095, ..., -0.06935462,
         0.02025251,  0.02485633],
       [-0.00356188,  0.01968796, -0.05759465, ...,  0.11565612,
        -0.00758806,  0.0375593 ]], shape=(182, 384), dtype=float32)

In [ ]:
vector_store.add_documents(chunks, embeddings)

In [ ]:
class RAGRetriever:

    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        self.vector= vector_store
        self.embedding_manager= embedding_manager

    def retriever(self,  query: str, top_k: int=5, score_threshold: float= 0.0)-> List[Dict[str, Any]]:
        query_embedding= self.embedding_manager.generate_embeddings([query])[0]

        try:
            result= self.vector.collection.query(
                query_embeddings= [query_embedding.tolist()],
                n_results= top_k
            )

            retrieved_docs=[]

            if result["documents"] and result["documents"] [0]:
                ids= result["ids"][0]
                documents= result["documents"][0]
                metadatas= result["metadatas"][0]
                distances= result["distances"][0]

                for i, (id,document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):

                    similarity_score= 1-distance

                    if similarity_score>=score_threshold:
                        retrieved_docs.append({
                            "id" : id,
                            "documents" : document,
                            "metadata" : metadata,
                            "distance" : distance,
                            "rank": i + 1
                        })

            else:
                print("Found Error!")

            return retrieved_docs

        except Exception as e:
            print(f"Error! {e}")
            return[]

In [ ]:
RagRetrive= RAGRetriever(vector_store, embedding_manager)
RagRetrive

In [ ]:
RagRetrive.retriever("State the properties of tree?")

Generating Embeddings of 1 texts
Embedding Done! Shape: (1, 384)


[{'id': 'doc_dd62f83f_112',
  'documents': 'P a g e | 7 DATA STRUCTURES \nDEPARTMENT OF CSE \nD, B, A. \n\uf0d8\uf020\nPROPERTIES OF THE TREE: \nAny node can be the root of the tree \n \nEvery node has a property there is exactly one path connect the root node to the particular \nnode. In a tree ,the root is identified is called rooted tree .If the root is not identified is called free \ntree Every node except the root node have the unique parent \n \nBINARY TREE: \n \nBinary tree is defined which is either empty or consists of a root node and disjoint sub tree called left \nchild and right child. The below diagram shows the binary tree. \n \n \n \nIn a binary tree no node can have more than two children. Each \nbinary tree is a tree, but not every tree is a binary tree. \nThe complete binary tree is defined as the intermediate nodes have a degree and leaf nodes are at the \nsame level. \n  \nThe below diagram shows the complete binary tree',
  'metadata': {'File_type': 'pdf',
   'keyw

In [ ]:
 ### Simple RAG pipeline with Groq LLM
from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv
load_dotenv()

### Initialize the Groq LLM (set your GROQ_API_KEY in environment)
groq_api_key = os.getenv("GROQ_API_KEY")

llm=ChatGroq(groq_api_key=groq_api_key,model_name="openai/gpt-oss-120b",temperature=0.1,max_tokens=1024)

## 2. Simple RAG function: retrieve context + generate response
def rag_simple(query,retriever,llm,top_k=3):
    ## retriever the context
    results = retriever.retriever(query, top_k=top_k)
    context="\n\n".join([doc['documents'] for doc in results]) if results else ""
    if not context:
        return "No relevant context found to answer the question."
    
    ## generate the answwer using GROQ LLM
    prompt=f"""Use the following context to answer the question concisely.
        Context:
        {context}

        Question: {query}

        Answer:"""
    
    response=llm.invoke([prompt.format(context=context,query=query)])
    return response.content

In [ ]:
answer=rag_simple("What is tree?",RagRetrive,llm)
print(answer)

Generating Embeddings of 1 texts
Embedding Done! Shape: (1, 384)
A **tree** is a non‑linear data structure consisting of a finite set of nodes arranged hierarchically. One node is designated as the root; every other node has exactly one parent, and nodes may have zero or more child nodes. The structure forms a collection of sub‑trees, with leaf nodes (no children) and internal (non‑terminal) nodes, connected by edges.
